# Suite EXT — Schema extension

A schema is extended by adding members to it: properties and adjacencies to an object schema, properties to a
relation, branches to a union (docs/SCHEMA_EXTENSION.md). A module extends schemas in its `install(txn)`, a transaction
the store runs (docs/OBJECT_IDENTITY.md, Decided, 86; docs/proposals/install). A schema the store was made holding, a
framework schema, only gains members, in any transaction; a store's own schemas change freely. Each store's framework
schemas are its own, so what one store's install adds, another lacks.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Paths, Plain, Validators
from support import entries_in, raises, spec_of


def crm(txn):
    """crm.Person, who knows persons and has tags; crm.Customer, refining it; crm.Vip, refining that; crm.Id, a union."""
    S = txn.Schemas
    return txn.uid("crm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Knows": S.OfRelation, "Person": S.OfObject, "Customer": S.OfObject, "Vip": S.OfObject, "Id": S.OfUnion})
        .spec([
            lambda _: _.Knows.name("crm.Knows").links([lambda t: t.name("a"), lambda t: t.name("b")]).uniques([]),  # a set
            lambda _: _.Person.name("crm.Person").ref(True).properties({
                "name": S.OfStr, "tags": S.OfContainer(lambda c: c.index([S.OfInt]).element([S.OfStr]))}).adjacencies({
                "knows": lambda a: a.relation([_.Knows]).me("a")}),
            lambda _: _.Customer.name("crm.Customer").ref(True).refines([_.Person]).properties({"account": S.OfStr}),
            lambda _: _.Vip.name("crm.Vip").ref(True).refines([_.Customer]).properties({"tier": S.OfInt}).adjacencies({
                "fans": lambda a: a.relation([_.Knows]).me("b")}),  # a refinement of a refinement
            lambda _: _.Id.name("crm.Id").branches([lambda t: t.name("n").schema([S.OfInt])]),
        ])]))])


store = B.Store()
store.transaction(crm)
Text, Int = store.Schemas.OfStr.as_schema(), store.Schemas.OfInt.as_schema()  # the store's own
Knows, Person, Customer, Id = (store.registered(name) for name in ("crm.Knows", "crm.Person", "crm.Customer", "crm.Id"))
EXTENDS = "an extension only adds members to"


def more(txn):
    """crm.Person's age and an anonymous end of crm.Knows, crm.Knows' since, crm.Id's text branch, and crm.Note."""
    S = txn.Schemas
    return txn.uid("crm.more/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Person": S.OfObject, "Knows": S.OfRelation, "Id": S.OfUnion, "Note": S.OfObject})
        .spec([
            lambda _: _.Person.name("crm.Person").properties({"age": S.OfInt}).adjacencies([
                lambda a: a.relation([_.Knows]).me("b")]),  # anonymous
            lambda _: _.Knows.name("crm.Knows").properties({"since": S.OfInt}),
            lambda _: _.Id.name("crm.Id").branches([lambda t: t.name("s").schema([S.OfStr])]),
            lambda _: _.Note.name("crm.Note").properties({"text": S.OfStr}),  # built: its own
        ])]))])


def domains(txn):
    """Domains, a scalar's domain and a quantity's unit: what mbse-expressions adds to a store, which extends its own
    `Schemas.OfScalar` and `Schemas.OfObject`."""
    S = txn.Schemas
    return txn.uid("domains/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Domain": S.OfObject, "HasDomain": S.OfRelation, "Scalar": S.OfObject, "Object": S.OfObject})
        .spec([
            lambda _: _.HasDomain.name("Expressions.Framework.HasDomain").links([
                lambda t: t.name("schema"), lambda t: t.name("domain")]).uniques([["schema"]]),
            lambda _: _.Domain.name("Expressions.Framework.Domain").ref(True).properties({"label": S.OfStr}).adjacencies({
                "schemas": lambda a: a.relation([_.HasDomain]).me("domain")}),
            lambda _: _.Scalar.name("Schemas.OfScalar").adjacencies({"domain": lambda a: a.relation([_.HasDomain]).me("schema")}),
            lambda _: _.Object.name("Schemas.OfObject").properties({"unit": S.OfStr}),
        ])]))])

## EXT-01 · An install adds members to the schemas a store holds; their objects and their refinements gain them

In [ ]:
with store.transaction():
    ada = store.crm.Person(lambda p: p.name("Ada"))  # built before the install
store.transaction(more)
assert list(Person.properties) == ["name", "tags", "age"] and len(Person.anonymous) == 1
assert list(Customer.properties) == ["name", "tags", "age", "account"]  # inherited
assert list(Knows.entry.properties) == ["since"] and [b.name for b in Id.branches] == ["n", "s"]
with store.transaction():
    assert ada.age.get() is None and store.crm.Person(ada, lambda p: p.age(36)).age.get() == 36  # absent until written
    assert store.crm.Customer(lambda c: c.name("Bo").age(7).account("A")).age.get() == 7
assert store.registered("crm.Note").validate() == []
store.transaction(more)  # run again: found, nothing written
assert list(Person.properties) == ["name", "tags", "age"] and len(Person.anonymous) == 1

## EXT-02 · A schema the store was made holding only gains members, each under a name it neither has nor a refinement of it declares; the extension check says what else a write would change

In [ ]:
with store.transaction():
    Kind = store.Schemas.OfObject(lambda o: (  # a store's own schema refining a framework schema
        o
        .name("crm.Kind")
        .refines([store.Schemas.OfObject])
        .properties([lambda t: t.name("badge").schema([Text])])
    ))
    store.Schemas.OfObject(lambda o: (  # and a refinement of that
        o
        .name("crm.SubKind")
        .refines([Kind])
        .adjacencies([lambda t: t.name("pins").relation([Knows]).me("a")])
    ))
OfObject, HasSchema = store.Schemas.OfObject.as_schema(), store.Schemas.HasSchema.as_schema()
for change, says in [
    (lambda: store.Schemas.OfObject(OfObject, lambda o: o.description("An object schema")),
     "'Schemas.OfObject': it changes 'description'"),
    (lambda: store.Schemas.OfObject(OfObject, lambda o: o.ref(False)), "'Schemas.OfObject': it changes 'ref'"),
    (lambda: store.Schemas.OfObject(OfObject, lambda o: o.properties(lambda adv: adv.remove(lambda e: e.name("ref")))),
     "'Schemas.OfObject': it removes or replaces what 'properties' holds"),
    (lambda: store.Schemas.OfObject(OfObject, lambda o: o.properties([lambda t: t.name("ref").schema([Int])])),  # retyped
     "'Schemas.OfObject': it removes or replaces what 'properties' holds"),
    (lambda: store.Schemas.OfRelation(HasSchema, lambda r: r.links([lambda t: t.name("c")])),
     "'Schemas.HasSchema': it adds to 'links', which an extension cannot"),
    (lambda: store.Schemas.OfObject(OfObject, lambda o: o.adjacencies([lambda t: t.name("ref").relation([Knows]).me("a")])),
     "'Schemas.OfObject': it adds 'ref', which it has"),
    (lambda: store.Schemas.OfObject(OfObject, lambda o: o.properties([lambda t: t.name("badge").schema([Text])])),
     "'Schemas.OfObject': it adds 'badge', which 'crm.Kind' declares"),
    (lambda: store.Schemas.OfObject(OfObject, lambda o: o.properties([lambda t: t.name("pins").schema([Text])])),
     "'Schemas.OfObject': it adds 'pins', which 'crm.SubKind' declares"),
    (lambda: store.Schemas.OfContainer(store.Schemas.OfRelation.as_schema().properties["uniques"].schema,
                                       lambda c: c.layout("sparse")),  # unnamed: its kind
     "a container schema: it changes 'layout'"),
]:
    with raises(TypeError, match=f"{EXTENDS} {says}"):
        with store.transaction():
            change()
assert "badge" not in OfObject.properties and OfObject.ref and OfObject.description is None  # refused: unwritten
with store.transaction():
    store.Schemas.OfObject(Person, lambda o: o.description("A person").ref(False))  # a store's own schema changes freely
assert Person.description == "A person" and not Person.ref

## EXT-03 · A framework schema gains members in any transaction, an imperative scope's or an install's, and changes no other way

In [ ]:
with store.transaction():
    store.Schemas.OfObject(store.Schemas.OfScalar, lambda o: o.properties([lambda t: t.name("unit").schema([Text])]))
    store.Schemas.OfObject(store.Schemas.OfScalar, lambda o: o.adjacencies([lambda t: t.relation([Knows]).me("b")]))  # anonymous
    with raises(TypeError, match=f"{EXTENDS} 'Schemas.OfScalar': it changes 'description'"):
        store.Schemas.OfObject(store.Schemas.OfScalar, lambda o: o.description("x"))
assert "unit" in store.Schemas.OfScalar.as_schema().properties and store.Schemas.OfScalar.as_schema().description is None
assert store.Schemas.OfScalar.as_schema().anonymous[-1].relation is Knows
assert B.OfScalar.Builder(store.Schemas.OfInt, False, store).unit.get() is None  # a builder of its schemas has it

## EXT-04 · What an install adds to a framework schema is its store's alone: another store holds its own; a meta-schema's new members are its schemas' builders'

In [ ]:
extended, plain = B.Store(), B.Store()
extended.transaction(domains)
assert "domain" in extended.Schemas.OfScalar.as_schema().adjacencies and "unit" in extended.Schemas.OfObject.as_schema().properties
assert "domain" not in plain.Schemas.OfScalar.as_schema().adjacencies and "Expressions.Framework.Domain" not in plain.names()
assert not extended.framework(extended.registered("Expressions.Framework.Domain"))  # what it builds is the store's own
with extended.transaction():
    Meters = extended.Schemas.OfObject(lambda o: (
        o
        .name("Meters")
        .properties([lambda t: t.name("value").schema([extended.Schemas.OfFloat])])
        .unit("m")
    ))
assert B.OfObject.Builder(Meters, False, extended).unit.get() == "m"
with extended.transaction():
    Feet = B.OfObject.Builder(Meters, store=extended).name("Feet").clone()
assert B.OfObject.Builder(Feet, False, extended).unit.get() == "m"  # a clone keeps it
assert B.OfObject.Builder(plain.Schemas.OfObject, False, plain).unit.get() is None  # another store's builders lack it
assert B.OfScalar.Builder(plain.Schemas.OfInt, False, plain).domain.get() is None


def trio(txn):
    S = txn.Schemas
    return txn.uid("trio/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"T3": S.OfRelation, "Scalar": S.OfObject})
        .spec([
            lambda _: _.T3.name("T3").links([lambda t: t.name("a"), lambda t: t.name("b"), lambda t: t.name("c")]),
            lambda _: _.Scalar.name("Schemas.OfScalar").adjacencies({"trio": lambda a: a.relation([_.T3]).me("a")}),
        ])]))])


with raises(TypeError, match="an extension's adjacency 'trio' is to a relation of two links and no properties, got 'T3'"):
    B.Store().transaction(trio)

## EXT-05 · An extension's relation links schema objects and data: one entry, held by both, written from either end and once in a snapshot; what a store lacking the extension ignores

In [ ]:
s = B.Store()
s.transaction(domains)
with s.transaction():
    int8 = s.Expressions.Framework.Domain(lambda d: d.label("int8"))
    Byte = s.Schemas.OfScalar(s.Schemas.OfInt, lambda x: x.domain([int8]))  # from the schema's end: the store's own scalar
    assert [e.schema is s.Schemas.OfInt.as_schema() for e in int8.schemas] == [True]  # within the transaction too
assert Byte is s.Schemas.OfInt.as_schema() and [e.schema is Byte for e in int8.schemas] == [True]  # seen from the data's end
with s.transaction():
    int16 = s.Expressions.Framework.Domain(lambda d: d.label("int16").schemas(lambda a: a.insert(lambda e: e.schema(Byte))))
assert [e.domain.label.get() for e in B.OfScalar.Builder(Byte, False, s).domain] == ["int8", "int16"]
assert Validators.Validate(s).Reachable(s.registered("Expressions.Framework.Domain"), int8) == []
log = Plain.ToPlain(s).Snapshot()
held = log[1]["mutations"][0]["upsert"]["exists"][0]["symbols"]  # the data, and the scalar its entries link
written = [e for symbol in held for name in ("schemas", "domain") for e in entries_in(spec_of(log, symbol, 1), name)]
assert len(written) == 2 and all("schema" in e for e in written)  # each once, from the data's end
back = B.Store()  # a round trip between stores with the same extensions
back.transaction(domains)
Plain.FromPlain(back).Snapshot(log)
assert len(list(B.OfScalar.Builder(back.Schemas.OfInt, False, back).domain)) == 2 and Plain.ToPlain(back).Snapshot() == log
with s.transaction():
    s.Schemas.OfScalar(Byte, lambda x: x.domain(lambda a: a.remove(lambda e: e.domain(int8))))
assert list(int8.schemas) == [] and len(list(int16.schemas)) == 1
# A store's basic scalar is its own: the entries linking it are the store's, and another store's is never linked.
one, other = B.Store(), B.Store()
one.transaction(domains)
other.transaction(domains)
with one.transaction():
    one.Expressions.Framework.Domain(lambda d: d.label("machine int").schemas(lambda a: a.insert(lambda e: e.schema(one.Schemas.OfFloat))))
seen = lambda store: [e.domain.label.get() for e in B.OfScalar.Builder(store.Schemas.OfFloat, False, store).domain]
assert seen(one) == ["machine int"] and seen(other) == []
third = B.Store()
third.transaction(domains)
Plain.FromPlain(third).Snapshot(Plain.ToPlain(one).Snapshot())
assert seen(third) == ["machine int"] and seen(other) == []
assert Plain.ToPlain(third).Snapshot() == Plain.ToPlain(one).Snapshot()  # read back as written
with one.transaction():
    with raises(TypeError, match="proxies can only link proxy instances, or schema objects their store holds"):
        one.Expressions.Framework.Domain(lambda d: d.schemas(lambda a: a.insert(lambda e: e.schema(other.Schemas.OfInt))))
# Read into a store without an extension, a sub-clause whose meaning needs it is ignored, and kept where asked: a
# property a schema the store was made holding lacks, an entry only such a schema could hold, a property of an entry of
# such a relation. The rest is read.
bare, ignored = B.Store(), []
Plain.FromPlain(bare).Snapshot(Plain.ToPlain(extended).Snapshot(), ignored)
assert [i["reason"] for i in ignored] == ["Schemas.OfObject has no property 'unit'"] * 2  # Meters', Feet's
assert all(i["path"].endswith(".as_property") for i in ignored) and list(bare.registered("Meters").properties) == ["value"]


def tagged(txn):
    """A tag a scalar has, declared from the scalar's end only."""
    S = txn.Schemas
    return txn.uid("tagged/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Tags": S.OfRelation, "Tag": S.OfObject, "Scalar": S.OfObject})
        .spec([
            lambda _: _.Tags.name("X.Tags").links([lambda t: t.name("scalar"), lambda t: t.name("tag")]),
            lambda _: _.Tag.name("X.Tag").ref(True),
            lambda _: _.Scalar.name("Schemas.OfScalar").adjacencies({"tags": lambda a: a.relation([_.Tags]).me("scalar")}),
        ])]))])


labels = B.Store()
labels.transaction(tagged)
with labels.transaction():
    labels.Schemas.OfScalar(labels.Schemas.OfInt, lambda x: x.tags([labels.X.Tag(lambda x: x)]))
    labels.Schemas.OfObject(lambda o: o.name("Note").properties([lambda t: t.name("text").schema([labels.Schemas.OfStr])]))
ignored = []
Plain.FromPlain(B.Store()).Snapshot(Plain.ToPlain(labels).Snapshot(), ignored)
assert [i["reason"] for i in ignored] == ["Schemas.OfScalar has no adjacency 'tags'"]  # written from the scalar's end
altered = Plain.ToPlain(labels).Snapshot()  # as an extension adding a property to Schemas.HasSchema would write it
text = next(m for m in spec_of(altered, "Note") if "as_adjacency" in m)["as_adjacency"]["mutations"][0]["insert"]["entries"][0]
typing = next(p for p in text["mutations"] if "as_link" in p)["as_link"]["mutations"][0]["set"]["def"]["mutations"]
typed = next(m for m in typing if "as_adjacency" in m and m["as_adjacency"]["name"] == "schema")
typed["as_adjacency"]["mutations"][0]["insert"]["entries"][0]["mutations"].append(
    {"as_property": {"name": "why", "mutations": [{"set": {"value": "sized"}}]}})
ignored = []
Plain.FromPlain(B.Store()).Snapshot(altered, ignored)
assert [i["reason"] for i in ignored] == [
    "an entry of 'schema' has no property 'why'", "Schemas.OfScalar has no adjacency 'tags'"]
assert Plain.FromPlain(B.Store()).Snapshot(altered)  # ignored, kept or not


def shaped(txn):
    """A scalar's shapes, object schemas, declared from the scalar's end only."""
    S = txn.Schemas
    return txn.uid("shaped/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Shapes": S.OfRelation, "Scalar": S.OfObject})
        .spec([
            lambda _: _.Shapes.name("X.Shapes").links([lambda t: t.name("scalar"), lambda t: t.name("shape")]),
            lambda _: _.Scalar.name("Schemas.OfScalar").adjacencies({"shapes": lambda a: a.relation([_.Shapes]).me("scalar")}),
        ])]))])


# An entry written from a store's own scalar, which a snapshot never writes, is written still, and what it links that has
# no name of its own follows it.
shapes = B.Store()
shapes.transaction(shaped)
with shapes.transaction():
    shape = shapes.Schemas.OfObject(lambda o: o.properties([lambda t: t.name("x").schema([shapes.Schemas.OfInt])]))  # unnamed
    shapes.Schemas.OfScalar(shapes.Schemas.OfInt, lambda x: x.shapes([shape]))
log = Plain.ToPlain(shapes).Snapshot()
again = B.Store()
again.transaction(shaped)
Plain.FromPlain(again).Snapshot(log)
assert len(list(B.OfScalar.Builder(again.Schemas.OfInt, False, again).shapes)) == 1 and Plain.ToPlain(again).Snapshot() == log
assert Plain.FromPlain(again).Snapshot(log) == [] and len(list(B.OfScalar.Builder(again.Schemas.OfInt, False, again).shapes)) == 1


def pinned(txn):
    """A board, a singleton, pinning schemas: data that links schema objects."""
    S = txn.Schemas
    return txn.uid("pinned/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Pins": S.OfRelation, "Board": S.OfObject})
        .spec([
            lambda _: _.Pins.name("X.Pins").links([lambda t: t.name("board"), lambda t: t.name("schema")]),
            lambda _: _.Board.name("X.Board").ref(True).singleton("x.Board").adjacencies({
                "pins": lambda a: a.relation([_.Pins]).me("board")}),
        ])]))])


# Read again, a snapshot adds nothing: what an entry defines where it is linked is made where the object it is written
# with held no such entry before (a singleton, which every store holding its schema has, or a store's own scalar).
boards = B.Store()
boards.transaction(pinned)
with boards.transaction():
    unnamed = [boards.Schemas.OfContainer(lambda c: c.index([boards.Schemas.OfInt]).element([element]))
               for element in (boards.Schemas.OfStr, boards.Schemas.OfInt)]
    boards.X.Board(boards.singleton("x.Board"), lambda b: b.pins(lambda a: a.insert(lambda e: e.schema(unnamed[0])).insert(
        lambda e: e.schema(unnamed[1]))))
log = Plain.ToPlain(boards).Snapshot()
again = B.Store()
again.transaction(pinned)
assert [made is again.singleton("x.Board") for made in Plain.FromPlain(again).Snapshot(log)] == [True]  # its data, the board
assert [made is again.singleton("x.Board") for made in Plain.FromPlain(again).Snapshot(log)] == [True]  # read again: found
assert len(again.singleton("x.Board").pins) == 2
assert [e.schema.element for e in again.singleton("x.Board").pins] == [again.Schemas.OfStr.as_schema(), again.Schemas.OfInt.as_schema()]


def objects_tagged(txn):
    """Tags an object schema has, declared from its end: any object schema's, the store's own included."""
    S = txn.Schemas
    return txn.uid("objects_tagged/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Tags": S.OfRelation, "Tag": S.OfObject, "Object": S.OfObject})
        .spec([
            lambda _: _.Tags.name("X.Tags").links([lambda t: t.name("object"), lambda t: t.name("tag")]),
            lambda _: _.Tag.name("X.Tag").ref(True),
            lambda _: _.Object.name("Schemas.OfObject").adjacencies({"tags": lambda a: a.relation([_.Tags]).me("object")}),
        ])]))])


# The store's own object schema, holding entries of the store's own relations, is written with its extension's alone.
own = B.Store()
own.transaction(objects_tagged)
with own.transaction():
    own.Schemas.OfObject(own.Schemas.OfObject, lambda x: x.tags([own.X.Tag(lambda t: t)]))
log = Plain.ToPlain(own).Snapshot()
assert [m["as_adjacency"]["name"] for m in spec_of(log, "Schemas.OfObject", 1) if "as_adjacency" in m] == ["tags"]
mirror = B.Store()
mirror.transaction(objects_tagged)
Plain.FromPlain(mirror).Snapshot(log)
assert len(list(B.OfObject.Builder(mirror.Schemas.OfObject, False, mirror).tags)) == 1 and Plain.ToPlain(mirror).Snapshot() == log